# 07. Unsupervised Anomaly Detection: Isolation Forest
## Intelligent Cyber Threat Detection & Network Traffic Analysis

---
### 1. Methodology: Normal Profiling vs. Zero-Day Detection
Supervised models excel at recognizing known attack patterns, but cannot classify zero-day exploits whose signatures have never been seen.
**Isolation Forest** constructs random partitioning trees on pure benign baseline communications. Outliers (anomalies) require fewer partitions to isolate.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd
from src.data_loader import load_dataset
from src.preprocessing import prepare_train_test_split
from src.anomaly_detection import NetworkAnomalyDetector

df, _ = load_dataset(use_sample_if_empty=True)
X_tr, X_te, y_tr, y_te, prep, _ = prepare_train_test_split(df, mode="binary")

# Train Isolation Forest strictly on benign training traffic
benign_tr = X_tr[y_tr == 0]
detector = NetworkAnomalyDetector(contamination=0.15)
detector.fit_benign_baseline(benign_tr)
print("Isolation Forest trained on benign baseline.")

### 2. Evaluating Detection Against Holdout Test Flows

In [ ]:
results = detector.evaluate_against_ground_truth(X_te, y_te)
print("Unsupervised Anomaly Detection Evaluation:")
for k, v in results.items():
    if k != "methodology_disclaimer":
        print(f"  {k}: {v}")

### 3. Critical Research Interpretation
> [!NOTE]
> An anomaly represents a statistical deviation from normal traffic. While it successfully flags unusual attack bursts, it also flags benign volume spikes. Anomaly detection is a triaging aid, not a definitive attack verdict.